# PS: 3C
Implement Anomaly Detection in Energy Consumption Data (Smart Grid
Dataset) using Autoencoder and build the model by using the following steps:
Steps:
  
  • Preprocess the data to fill any missing values, normalize, and create time windows for
sequential input.
  
  • Use an LSTM-based autoencoder to learn temporal patterns in normal consumption.
  
  • Set an error threshold and detect anomalies when the reconstruction error exceeds this
threshold.
  
  Dataset: The Smart Grid dataset records electricity consumption across multiple timestamps
for various households.
  
  Goal: Detect unusual consumption patterns by training an autoencoder on typical (normal)
energy usage patterns

In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow.keras import models, layers


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "C:\Program Files\Python312\Lib\runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Program Files\Python312\Lib\runpy.py", line 88, in _run_code
    exec(code, run_globals)
  File "C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  Fil

AttributeError: _ARRAY_API not found

In [9]:
data = pd.read_csv("energy consumption.csv")

print(data.head())
print(data.shape)

          Timestamp  Electricity_Consumed  Temperature  Humidity  Wind_Speed  \
0  01-01-2024 00:00              0.457786     0.469524  0.396368    0.445441   
1  01-01-2024 00:30              0.351956     0.465545  0.451184    0.458729   
2  01-01-2024 01:00              0.482948     0.285415  0.408289    0.470360   
3  01-01-2024 01:30              0.628838     0.482095  0.512308    0.576241   
4  01-01-2024 02:00              0.335974     0.624741  0.672021    0.373004   

   Avg_Past_Consumption  Anomaly_Label  
0              0.692057              0  
1              0.539874              0  
2              0.614724              0  
3              0.757044              0  
4              0.673981              0  
(5000, 7)


In [7]:
# SEPARRATE x AND Y
X = df.drop(columns = ['Timestamp', 'Anomaly_Label'])
y = df['Anomaly_Label']

In [10]:
# PREPROCESSING
data = data.fillna(data.mean(numeric_only = True))

x = data.drop(['Timestamp', 'Anomaly_Label'], axis=1)
y = data['Anomaly_Label']

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

In [11]:
# KEEP ONLY NOMRMAL DATA FOR TRAINING
normal = x_train[y_train==0]


In [12]:
# Normalize data
scaler = StandardScaler()
normal = scaler.fit_transform(normal)
x_test = scaler.transform(x_test)

In [13]:
# Split normal data into training and validation
x_train, x_val = train_test_split(
    normal, test_size=0.1, random_state=42
)

In [15]:
# Reshape data for LSTM
x_train = x_train.reshape(-1, x_train.shape[1], 1)
x_val = x_val.reshape(-1, x_val.shape[1], 1)
x_test = x_test.reshape(-1, x_test.shape[1], 1)


In [16]:
# Step 4: Build LSTM Encoder and Decoder

encoder = models.Sequential([
    layers.Input(shape=(x_train.shape[1], 1)),
    layers.LSTM(16)
])

decoder = models.Sequential([
    layers.Input(shape=(16,)),
    layers.RepeatVector(x_train.shape[1]),
    layers.LSTM(16, return_sequences=True),
    layers.TimeDistributed(layers.Dense(1))
])

model = models.Sequential([encoder, decoder])

In [17]:
# Step 5: Compile model
model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

In [18]:
# Step 6: Train model
model.fit(
    x_train, x_train,
    validation_data=(x_val, x_val),
    epochs=20,
    batch_size=512
)

Epoch 1/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 8s 256ms/step - loss: 0.9595 - mae: 0.7852 - val_loss: 0.9438 - val_mae: 0.7734
Epoch 2/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - loss: 0.9337 - mae: 0.7745 - val_loss: 0.9185 - val_mae: 0.7626
Epoch 3/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - loss: 0.9059 - mae: 0.7626 - val_loss: 0.8907 - val_mae: 0.7506
Epoch 4/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - loss: 0.8754 - mae: 0.7493 - val_loss: 0.8608 - val_mae: 0.7372
Epoch 5/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 50ms/step - loss: 0.8437 - mae: 0.7348 - val_loss: 0.8323 - val_mae: 0.7234
Epoch 6/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - loss: 0.8171 - mae: 0.7230 - val_loss: 0.8126 - val_mae: 0.7139
Epoch 7/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - loss: 0.8036 - mae: 0.7169 - val_loss: 0.8070 - val_mae: 0.7105
Epoch 8/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - loss: 0.8009 - mae: 0.7155 - val_loss: 0.8052 - val_mae: 0.7092
Epoch 9/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - loss: 0.7978 - mae: 0.7137 -

In [19]:
# Step 7: Calculate threshold
train_pred = model.predict(x_val)

train_error = np.mean(
    np.square(x_val - train_pred), axis=(1, 2)
)

threshold = np.percentile(train_error, 99)

print("Threshold:", threshold)

12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 62ms/step
Threshold: 2.7208541768895884


In [20]:
# Step 8: Detect anomalies
test_pred = model.predict(x_test)

test_error = np.mean(
    np.square(x_test - test_pred), axis=(1, 2)
)

prediction = (test_error > threshold).astype(int)

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step


In [21]:
# Step 9: Evaluate model
print("Confusion Matrix:")
print(confusion_matrix(y_test, prediction))

print("\nClassification Report:")
print(classification_report(y_test, prediction))

print("Number of anomalies detected:", prediction.sum())

Confusion Matrix:
[[949   1]
 [ 43   7]]

Classification Report:
              precision    recall  f1-score   support

           0       0.96      1.00      0.98       950
           1       0.88      0.14      0.24        50

    accuracy                           0.96      1000
   macro avg       0.92      0.57      0.61      1000
weighted avg       0.95      0.96      0.94      1000

Number of anomalies detected: 8
